# 07 — YOLOv8 training and model saving

This notebook trains a YOLOv8 detector on the dataset prepared in notebook 06
and saves the artefacts needed to reuse it.

It:

1. reads `data/yolo/data.yaml` and `data/yolo/train_config.json`;
2. selects training hyperparameters adapted to the available hardware;
3. **trains** `yolov8n` with the augmentation policy of notebook 06;
4. plots the loss and metric curves read from `results.csv`;
5. validates the trained model on the validation split;
6. saves the best weights, curves and a JSON summary under `models/yolo/`.

> **How to read this notebook:** every code cell is preceded by a short
> summary that explains what the step does and how to interpret its output.

## 1. Imports and paths

Imports Ultralytics, PyTorch (to detect the device) and plotting utilities,
then defines where runs and final artefacts are written.

**How to read the output:** the printed Ultralytics and PyTorch versions, plus
the model directory. A CUDA-enabled GPU is strongly recommended; on CPU the
training runs, but it is much slower.

In [1]:
from pathlib import Path
import json
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_YAML = ROOT_DIR / "data" / "yolo" / "data.yaml"
TRAIN_CONFIG_PATH = ROOT_DIR / "data" / "yolo" / "train_config.json"
MODELS_DIR = ROOT_DIR / "models" / "yolo"

RUN_NAME = "brain_tumor_yolov8n"
PRETRAINED_PATH = MODELS_DIR / "yolov8n.pt"
SAVED_MODEL_PATH = MODELS_DIR / "brain_tumor_yolov8n.pt"

for directory in (MODELS_DIR, ):
    directory.mkdir(parents=True, exist_ok=True)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch    :", torch.__version__)
print("Models dir :", MODELS_DIR)

Ultralytics: 8.4.188
PyTorch    : 2.14.1+cu130
Models dir : /home/fangyuan/Projects/Diagnostic-des-tumeurs/models/yolo


## 2. Load the dataset configuration and choose a resource profile

Reads `data.yaml` and the augmentation/optimization config written by
notebook 06, then checks whether a GPU is available.

The resource profile is chosen automatically:

| device | epochs | imgsz | batch | patience |
|--------|-------:|------:|------:|---------:|
| GPU    | 100    | 640   | 32    | 20       |
| CPU    | 10     | 320   | 8     | 5        |

On CPU the image size and batch are reduced to keep the run tractable
(roughly 5 minutes per epoch on a laptop); on GPU the full resolution is used.
You can override these values after reading the printed summary.

**How to read the output:** the dataset root, split paths, class names, device
and selected profile. `CUDA available: True` means the GPU profile is used.

In [ ]:
for path in (DATA_YAML,):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run 06_yolo_preprocessing.ipynb first."
        )

with open(DATA_YAML) as handle:
    data_config = yaml.safe_load(handle)


cuda_available = torch.cuda.is_available()
if cuda_available:
    device = 0
    device_name = torch.cuda.get_device_name(0)
    profile = {
        "epochs": 100, 
        "imgsz": 640, 
        "batch": 32, 
        "workers": 8, 
        "patience": 20
    }
else:
    device = "cpu"
    device_name = "CPU"
    profile = {"epochs": 10, "imgsz": 320, "batch": 8, "workers": 4, "patience": 5}

print("Dataset root :", data_config["path"])
print(
    "Train / val / test:",
    data_config["train"],
    "/",
    data_config["val"],
    "/",
    data_config["test"],
)
print("Classes (%d) :" % data_config["nc"], data_config["names"])
print("CUDA available:", cuda_available, "| device:", device_name)
print("Profile      :", profile)

## 3. Build the final training arguments

Merges the resource profile with the optimization and augmentation parameters
of notebook 06 into the dictionary passed to `model.train(...)`.

In [2]:
train_args = {
    "data": str(DATA_YAML),
    "epochs": profile["epochs"],
    "imgsz": profile["imgsz"],
    "batch": profile["batch"],
    "workers": profile["workers"],
    "patience": profile["patience"],
    "device": device,
    "name": RUN_NAME,
    "exist_ok": True,
    "seed": 42,
}

NameError: name 'profile' is not defined

## 4. Reuse or download the pretrained weights

YOLOv8 starts from COCO-pretrained weights (`yolov8n.pt`), which is what makes
transfer learning possible on a small medical dataset. The file is downloaded
once and cached in `models/yolo/` so later runs are offline-friendly.

**How to read the output:** `Using cached weights` on the second run; the
download message and file size on the first run. A 6 MB file is expected for
the `n` (nano) variant.

In [ ]:
if PRETRAINED_PATH.exists():
    print("Using cached weights:", PRETRAINED_PATH)
else:
    from ultralytics.utils.downloads import attempt_download_asset

    downloaded = Path(attempt_download_asset("yolov8n.pt")).resolve()
    shutil.copy2(downloaded, PRETRAINED_PATH)
    # attempt_download_asset saves to the current directory: clean it up.
    if downloaded != PRETRAINED_PATH.resolve() and downloaded.parent == ROOT_DIR.resolve():
        downloaded.unlink(missing_ok=True)
    print("Downloaded weights:", PRETRAINED_PATH)

print("Size:", round(PRETRAINED_PATH.stat().st_size / 1e6, 2), "MB")

## 5. Train YOLOv8

Launches `model.train(...)` with the arguments of step 3.

Depending on the device this cell is long: on GPU expect a few minutes per
epoch, on CPU much more. The progress bar shows the losses; do not interrupt
it unless the losses stop improving.

**How to read the output:** `box_loss`, `cls_loss` and `dfl_loss` should
decrease. At the end Ultralytics prints the validation metrics and the path of
the run directory.

In [ ]:
model = YOLO(str(PRETRAINED_PATH))
results = model.train(**train_args)

run_dir = Path(model.trainer.save_dir)
print("Run directory:", run_dir)
print("Final metrics:", results.results_dict)

## 6. Plot the training curves

Reads `results.csv` from the run directory and plots the three losses
(box regression, classification, distribution focal loss) and the four
validation metrics (precision, recall, mAP@0.5, mAP@0.5:0.95).

**How to read the output:** all losses should go down and the metrics up. A
validation loss that rises while the training loss falls indicates
over-fitting; a flat mAP means the model stopped learning and the run could be
stopped earlier.

In [ ]:
history = pd.read_csv(run_dir / "results.csv")
history.columns = [column.strip() for column in history.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for column in ["train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss", "train/dfl_loss", "val/dfl_loss"]:
    if column in history:
        axes[0].plot(history["epoch"], history[column], label=column)
axes[0].set_title("Detection losses")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)

for column in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
    if column in history:
        axes[1].plot(history["epoch"], history[column], label=column)
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

display(history.tail(3))

## 7. Validate the trained model on the validation split

Runs `model.val(split="val")` on the best checkpoint. Validation uses the
`val` path of `data.yaml` (`images/valid`) and applies the same preprocessing
as training, without augmentation.

**How to read the output:** the four detection metrics
(`precision`, `recall`, `mAP50`, `mAP50-95`) and the per-class values.
mAP@0.5 measures detection at a loose overlap threshold, mAP@0.5:0.95 averages
over stricter thresholds and is the harder score.

In [ ]:
val_metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=train_args["imgsz"],
    batch=train_args["batch"],
    device=device,
    name=f"{RUN_NAME}_val",
    exist_ok=True,
    plots=True,
)

print(json.dumps(val_metrics.results_dict, indent=2))

## 8. Per-class validation metrics

Builds a table with precision, recall, AP@0.5 and AP@0.5:0.95 for every class
present in the validation set.

**How to read the output:** a class with a low recall is a class the detector
misses; a low precision means it produces boxes on healthy regions. Compare the
classes with the smallest number of boxes (see notebook 05) — rare classes are
usually the weakest.

In [ ]:
def per_class_table(metrics):
    rows = []
    for index, class_id in enumerate(metrics.box.ap_class_index):
        rows.append(
            {
                "class": metrics.names[int(class_id)],
                "precision": float(metrics.box.p[index]),
                "recall": float(metrics.box.r[index]),
                "mAP50": float(metrics.box.ap50[index]),
                "mAP50-95": float(metrics.box.ap[index]),
            }
        )
    return pd.DataFrame(rows).sort_values("mAP50-95", ascending=False, ignore_index=True)


val_table = per_class_table(val_metrics)
display(val_table)

## 9. Save the trained model and artefacts

Copies the best checkpoint and the useful run files to a stable location:

```
models/yolo/
├── brain_tumor_yolov8n.pt      # best weights, used by notebook 08
├── training_summary.json       # metrics, paths and counts
├── training_args.yaml          # exact arguments of the run
├── training_results.csv        # one row per epoch
└── training_plots/             # curves and dataset plots
```

**How to read the output:** the path of every copied file. Notebook 08 loads
`brain_tumor_yolov8n.pt`, so this cell is what makes the evaluation notebook
independent from the run directory.

In [ ]:
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.exists():
    best_weights = run_dir / "weights" / "last.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"No weights found in {run_dir / 'weights'}")

shutil.copy2(best_weights, SAVED_MODEL_PATH)
shutil.copy2(run_dir / "args.yaml", MODELS_DIR / "training_args.yaml")
shutil.copy2(run_dir / "results.csv", MODELS_DIR / "training_results.csv")

plots_dir = MODELS_DIR / "training_plots"
plots_dir.mkdir(exist_ok=True)
for plot_name in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "labels.jpg",
]:
    plot_path = run_dir / plot_name
    if plot_path.exists():
        shutil.copy2(plot_path, plots_dir / plot_name)

summary = {
    "created_by": "07_yolo_training.ipynb",
    "model": "yolov8n",
    "weights": str(SAVED_MODEL_PATH.relative_to(ROOT_DIR)),
    "run_dir": str(run_dir.relative_to(ROOT_DIR)),
    "dataset": str(DATA_YAML.relative_to(ROOT_DIR)),
    "classes": data_config["names"],
    "device": device_name,
    "epochs_requested": train_args["epochs"],
    "epochs_completed": int(history["epoch"].max()) + 1,
    "imgsz": train_args["imgsz"],
    "batch": train_args["batch"],
    "validation_metrics": {key: float(value) for key, value in val_metrics.results_dict.items()},
    "per_class_validation": val_table.to_dict(orient="records"),
}

with open(MODELS_DIR / "training_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

print("Saved weights :", SAVED_MODEL_PATH)
print("Saved summary :", MODELS_DIR / "training_summary.json")
print("Saved plots   :", plots_dir)

## Next step

Open **`08_yolo_evaluation.ipynb`**. It loads
`models/yolo/brain_tumor_yolov8n.pt`, evaluates it on the **test** split,
compares validation and test performance to judge generalization, inspects the
predictions on sample images and saves the evaluation artefacts.